# Lecture 6 - Data Workflow & Automation

Today we will continue with pandas basics. And then take a look a small example of data workflow that simulates a data pipeline and it can be automated further to handle tasks repetitively.

## Pandas Basics - Part 2

First, we continue with pandas basics.

**What we'll cover:**

1. Adding columns
7. Filtering rows
8. Grouping and aggregating
9.  Sorting
10. Date & Time

### Recap - pandas Basics Part 1

1. The two core pandas objects: `Series` (1D) and `DataFrame` (2D)
   - One row from a `DataFrame` is a `Series`
   - One column from a `DataFrame` is also a `Series`
   - One cell sometimes is referred as an element in pandas documentation. 
2. Loading data — from a dict, CSV, and Excel
   - We created a DataFrame by input a <font color=Lime> clean </font> restaurant dataset with code
   - pd.to_csv() the DataFrame to a data folder with relative path
   - pd.read_csv() to import the data again.
3. Inspecting a DataFrame with methods.
4. Selecting columns and rows.
   - Column: think basic data types dictionary and list
   - Row : loc or iloc [row]
   - Cell: loc or iloc [row, column]
5. Cleaning data — the pandas way
   - We created a DataFrame with <font color=Fuchsia> raw </font> data

Let's pick it up from here. 

In [2]:
# raw (dirty) dataset from 5 cleaning data - the pandas way

import pandas as pd

# Raw data
raw_data = [
    {"restaurant": "  pasta palace   ",  "city": "Stockholm",  "revenue": "182000",    "currency": "SEK", "covers": "210", "rating": "4.5"},
    {"restaurant": "Burger Barn  ",   "city": "Stockholm",  "revenue": "95000",     "currency": "SEK", "covers": "180", "rating": "3.8"},
    {"restaurant": "SUSHI SPOT",      "city": "Copenhagen", "revenue": "28000",     "currency": "DKK", "covers": "95",  "rating": "4.9"},
    {"restaurant": "taco town",       "city": " Oslo ",       "revenue": "61000",     "currency": "NOK", "covers": "140", "rating": None},
    {"restaurant": "Pasta Palace",    "city": "Stockholm",  "revenue": "182000",    "currency": "SEK", "covers": "210", "rating": "4.5"},
    {"restaurant": "Green Garden",    "city": "Oslo",       "revenue": "see notes", "currency": "NOK", "covers": "88",  "rating": "4.1  "},
    {"restaurant": "Burger Barn",     "city": "Copenhagen", "revenue": "31500",     "currency": "DKK", "covers": "210", "rating": "6.0"},
    {"restaurant": "  Ramen House ",  "city": "Stockholm",  "revenue": "74000",     "currency": "SEK", "covers": None,  "rating": "4.2"},
]

raw = pd.DataFrame(raw_data)

In [ ]:
raw.to_csv("data/restaurants_raw.csv", index=False)

### Inpect Raw Data

This DataFrame has no numeric columns before cleansing.

In [ ]:
# 3. inpect a DataFrame (exploratory data analysis - EDA)

print("Shape:", raw.shape)

print("\nColumns:")
print(raw.dtypes)

print("\n--- first (3) rows ---")
print(raw.head(3))

print("\n--- last (3) rows ---")
print(raw.tail(3))

In [ ]:
# compare ouptut here with lecture 5's clean.describe()
# TODO: read help(pd.DataFrame.describe)

print("\n--- Summary when all columns are strings ---")
print()
print(raw.describe())

print("\n--- Missing values: ---")
print(raw.isna().sum())

# Show data side by side for easy reading
raw

### Introducing `pandas.DataFrame.copy` 

Suppose that we are assisting larger and more complex data projects where we will perform exploration, analysis, data modeling (as for database, data warehouse, lakehouse etc.) *BEFORE* or *WHILE* the solution designs are taking place. During these processes, it helps to simulate how the data will be carried through later on in production - therefore, *HOW* we do sometimes can simulate an actual data workflow that later on getting scaled up for implementation and deployment. 

> **BEFORE** as often referred as *from 0 to 1*. <br>
> **WHILE** as actual agile development is happening in contrast of more traditional waterfall method. <br>
> **HOW** here can be referred as design-led product thinking and test-driven development, meaning we disect larger problems to manageable and **verifiable** smaller pieces to prototype in order to understand both the problems and the solutions better. 

We had a glimpse upon a mini example of such data workflow in lecture 3 (Week 14: 14.1 Control flow - A first look at data workflow, W14 Reference - a data workflow). Python is quite suitable for this type of work - I hope by now you have built your early understanding how this might work with code interacting with physical data at your finger tips. 

Keeping the above in mind, we will now adjust how 5. cleaning data - the pandas way will be carried out: **we will isolate the raw data from its downstream transformation** - a bit like how we are treating bronze to silver layer transformation in the medallion architecture. Remember when we drew on a whiteboard and compared to the mini data workflow example? If not, read [what is the medallion lakehouse architecture](https://learn.microsoft.com/en-us/azure/databricks/lakehouse/medallion) for more info.

With pandas, we simulate the bronze to silver layer transformation by starting with ``pandas.DataFrame.copy`` method. 

There is another reason besides data workflow good practice to warrant an indepth introduction for a *mere copy* method. 

This is because Python's native *copy* behavior works quite surprisingly different. Read this Medium article about [understanding variable assignment, shallow copy and deep copy in Python: explained for beginners](https://medium.com/@send2anshup/understanding-variable-assignment-shallow-copy-and-deep-copy-in-python-explained-for-beginners-c275fc4def95). The article also explained a few real-world example how these default features are needed for and used.

For us, we will take pandas copy method, which by default is ``deepcopy``. Run help to read for yourself. Below are quoted from ``help(pd.DataFrame.copy)``:

>Help on function copy in module pandas.core.generic:
>
>copy(self, deep: bool = True) -> Self<br>
>&emsp; Make a copy of this object's indices and data.<br>
>
>&emsp; When ``deep=True`` (default), a new object will be created with  <br>
>&emsp; a copyof the calling object's data and indices. <br>
>&emsp; <font color=lime> Modifications </font> to the data or indices of the copy <br>
>&emsp; will <font color=lime> not be reflected </font> in the <font color=lime> original object </font> (see notes below).

So, let's get to it.

In [ ]:
# this is a variable assignment pointing to the same underlying object
# does not isolate raw from raw_1 (an example name for transformation 1)

raw_1 = raw

# make some changes to raw_1 now and call both to compare output

In [3]:
# this is a pandas.DataFrame.copy
# now that raw must be a DataFrame object to "trigger" pandas copy method. A list copy for example is a Python default shallow copy

raw_1 = raw.copy()

### 5. Cleansing data with `map` and `apply`

Compare below with lecture 5 same section. Do you see how they differ?

In [4]:
# 5a. fix multiple string columns

raw_1[["restaurant", "city"]] = (raw_1[["restaurant", "city"]]
                                   .map(lambda x: x.strip().capitalize()))

# 5b & 5c. convert to numeric with apply and chain in fillna for missing data

raw_1[["revenue","covers"]] = (raw_1[["revenue","covers"]]
                               .apply(pd.to_numeric,errors="coerce")
                               .fillna(0))

# 5b & 5c. convert to numeric and handle irregular values
# Rating with NaN is probably meaningful for this business case. Here we choose to keep it as such instead of replacing with 0.

raw_1["rating"]  = (pd.to_numeric(raw_1["rating"],  errors="coerce")
                    .clip(upper=5.0))

# 5d: dedup when certain they are duplicates

raw_1 = raw_1.drop_duplicates(subset=["restaurant", "city"], keep="first")

raw_1

,restaurant,city,revenue,currency,covers,rating
0,Pasta palace,Stockholm,182000.0,SEK,210.0,4.5
1,Burger barn,Stockholm,95000.0,SEK,180.0,3.8
2,Sushi spot,Copenhagen,28000.0,DKK,95.0,4.9
3,Taco town,Oslo,61000.0,NOK,140.0,NaN
5,Green garden,Oslo,0.0,NOK,88.0,4.1
6,Burger barn,Copenhagen,31500.0,DKK,210.0,5.0
7,Ramen house,Stockholm,74000.0,SEK,0.0,4.2


In [ ]:
# Compare to raw

raw

### Pandas Function Application

>Quoted from pandas documentation **function application**: to apply your own or another library’s functions to pandas objects, you should be aware of the three methods below. The appropriate method to use depends on whether your function expects to operate on an entire ``DataFrame`` or ``Series``, row- or column-wise, or elementwise.

- Applying Elementwise (cell) Functions: ``map()``
- Row or Column-wise Function Application: ``apply()``
- Tablewise Function Application: ``pipe()``

The official documentation is actually one of the best sources to read and learn about them. Please visit page: https://pandas.pydata.org/pandas-docs/dev/user_guide/basics.html#function-application

We have a few examples so it's earlier for you to relate to the docs. 

### A Refresher on Jupyter Variables

In lecture 1 (13.1.2 Python basics - The four building blocks), we introduced  variables. ALso how to list all, delete specific one or multiple variables. Now it's good time to revisit that section.

In addition, we can also use an area we haven't been using so often: terminal > JUPYTER VARIABLES. 

Toggle your terminal and find JUPYTER to have a look now. 

We will keep an eye on this this when we start to transform the dataset. 

### 6. Adding columns

If prefered, now can be good time to create next copy to isolate cleansing phase from transformation, such as adding columns or removing columns. Depending on the data warehouse or lakehouse architecture principle, this phase can be considered silver layer engineering work. 

Don't worry if you are uncertain at the moment. The rule of thumb is to keep your own workflow easy to understand, debug and improve. 

In compex projects, it normally becomes quite natural where to break and set next variable before continue. In SQL, complex transformation sometimes contain multiple temporary tables organized like a hand-over chain in one script to perform what we do here. This is a comparable approach as we do here - while we work almost like a "sandbox" style with Python and Jupyter Notebook.

In [5]:
raw_2 = raw_1.copy()

Next, we want to convert revenues to the same currency SEK. We do so by adding two columns by adding a calcuated SEK amount as a new column. 

In [6]:
# For currency conversion we create a FX (foreign exchange) dictionary - a simple mapping table.

EXCHANGE_RATES = {"SEK": 1.0, "DKK": 1.45, "NOK": 0.98}

# new column with FX rate
raw_2["rate"] = raw_2["currency"].map(EXCHANGE_RATES)

# new calculated column for revenue in SEK
raw_2["revenue_sek"] = raw_2["revenue"] * raw_2["rate"]

raw_2

,restaurant,city,revenue,currency,covers,rating,rate,revenue_sek
0,Pasta palace,Stockholm,182000.0,SEK,210.0,4.5,1.00,182000.0
1,Burger barn,Stockholm,95000.0,SEK,180.0,3.8,1.00,95000.0
2,Sushi spot,Copenhagen,28000.0,DKK,95.0,4.9,1.45,40600.0
3,Taco town,Oslo,61000.0,NOK,140.0,NaN,0.98,59780.0
5,Green garden,Oslo,0.0,NOK,88.0,4.1,0.98,0.0
6,Burger barn,Copenhagen,31500.0,DKK,210.0,5.0,1.45,45675.0
7,Ramen house,Stockholm,74000.0,SEK,0.0,4.2,1.00,74000.0


In [7]:
# New column revenue per cover — pandas handles division column-by-column
# Where covers is NaN or 0, the result is NaN automatically (no ZeroDivisionError)

raw_2["rev_per_cover"] = (raw_2["revenue_sek"] / raw_2["covers"]).round(1)

For the next new column revenue_tier, we want to label rows (restaurants) based on the size of revenue. We will look at three alternative ways to get this done.

In [ ]:
# Option 1: the pandas native pandas.cut function

# first without label the range (interval)
raw_2["revenue_tier"] = pd.cut(raw_2["revenue_sek"],
                               bins=[0, 50000, 100000, float("inf")]
                               )
raw_2

In [8]:
# clearly labeled

raw_2["revenue_tier"] = pd.cut(raw_2["revenue_sek"],
                               bins=[-float("inf"), 50000, 100000, float("inf")],
                               labels=["Low", "Medium", "High"])

raw_2

,restaurant,city,revenue,currency,covers,rating,rate,revenue_sek,rev_per_cover,revenue_tier
0,Pasta palace,Stockholm,182000.0,SEK,210.0,4.5,1.00,182000.0,866.7,High
1,Burger barn,Stockholm,95000.0,SEK,180.0,3.8,1.00,95000.0,527.8,Medium
2,Sushi spot,Copenhagen,28000.0,DKK,95.0,4.9,1.45,40600.0,427.4,Low
3,Taco town,Oslo,61000.0,NOK,140.0,NaN,0.98,59780.0,427.0,Medium
5,Green garden,Oslo,0.0,NOK,88.0,4.1,0.98,0.0,0.0,Low
6,Burger barn,Copenhagen,31500.0,DKK,210.0,5.0,1.45,45675.0,217.5,Low
7,Ramen house,Stockholm,74000.0,SEK,0.0,4.2,1.00,74000.0,inf,Medium


In [ ]:
# What is float("inf")? Try remove float("inf") or replace it with say 120000 and see what happens.

raw_2["revenue_tier"] = pd.cut(raw_2["revenue_sek"],
                               bins=[0, 50000, 100000, 120000],
                               labels=["Low", "Medium", "High"])

raw_2

In [ ]:
# Option 2: a custom function AND .apply()

def revenue_tier(rev):
    if rev >= 100000:
        return "High"
    elif rev >= 50000:
        return "Medium"
    else:
        return "Low"

raw_2["tier"] = raw_2["revenue_sek"].apply(revenue_tier)

In [ ]:
# Option 3: with a lambda and .apply. 

# below can be written in one line. I personally find it hard to read - but maybe it works after seeing it often:) I reformatted the code, slightly easier to read.

# raw_2["tier"] = raw_2["revenue_sek"].apply(lambda x:"High" if x >= 100000 else "Medium" if x >= 50000  else "Low")

raw_2["tier"] = (raw_2["revenue_sek"]
                 .apply(lambda x: 
                        "High" if x >= 100000 
                        else "Medium" if x >= 50000 
                        else "Low"))

### 7. Filtering Rows & Boolean Indexing

In 2. Inspect a DataFrame, we have seen isna()

In [ ]:
raw_2.isna()

This boolean method is used for filtering rows quickly in DataFrame. This functionality is also called **boolean masking** or **boolean indexing**: when the condition is met (True/False), filter rows.

> The syntax maps directly to SQL `WHERE` clauses.

In [ ]:
raw_2[raw_2["revenue_sek"] > 50000]

In [ ]:
raw_2[raw_2["city"] == "Stockholm"]

In [ ]:
# Combine conditions with & (and) and | (or)
# Note: use parentheses around each condition — Python operator precedence requires it

# Stockholm restaurants with revenue above 80 000 SEK
raw_2[(raw_2["city"] == "Stockholm") & (raw_2["revenue_sek"] > 80000)]

In [ ]:
# Filter out rows where rating is missing
# .notna() is the opposite of .isna()

raw_2[raw_2["rating"].notna()]

## BREAK TIME 🐍

>Why did the data analyst break up with the automation script?<br>
>*It kept running without her.*

### 8. Grouping and aggregating `groupby`

> In short: this is the DataFrame equivalent of SQL `GROUP BY`.


In [ ]:
raw_2.groupby("city")["revenue_sek"].sum()

In [ ]:
type(raw_2.groupby("city")["revenue_sek"].sum())

In [ ]:
raw_2.groupby("city")["revenue_sek"].sum().index

In [ ]:
# If I want to create a new DataFrame out of the above Series

# Option 1: Directly in groupby() but specify as_index=False tells groupby to not make city the index, returning a DataFrame directly.

city_revenue = (raw_2
                .groupby("city", as_index=False)["revenue_sek"]
                .sum())

city_revenue

In [ ]:
# Option 2: use reset_index as a chain

city_revenue_2 = (raw_2
                .groupby("city")["revenue_sek"]
                .sum()
                .reset_index())

city_revenue_2

In [ ]:
# Option 3: to_frame()

city_revenue_3 = (raw_2.groupby("city")["revenue_sek"]
                .sum()
                .to_frame())
city_revenue_3

# this use city as index - if you'd like that

In [ ]:
del city_revenue_2, city_revenue_3

### Now We Have Pivot Table

In [ ]:
# If we add more columns to group by to aggregate several columns, we got a variation of pivot table.

raw_2.groupby(["city","currency"])[["revenue_sek","covers"]].sum()

In [ ]:
# Another pivot table

raw_2.pivot_table(index="city", 
                  columns="revenue_tier",
                  values="rating",
                  aggfunc="count")

In [ ]:
# I wanted to show the revenue_tier in reversed order and found out I had to do this. There is no way I will remember the iloc...

raw_2.pivot_table(index="city", 
                  columns="revenue_tier",
                  values="rating",
                  aggfunc="count").iloc[:, ::-1]

In [ ]:
# Then perhaps we do this instead:

def reverse_columns(df):
    return df.iloc[:, ::-1]

In [ ]:
# Can we chain it?

raw_2.pivot_table(index="city", 
                  columns="revenue_tier",
                  values="rating",
                  aggfunc="count").reverse_columns


In [ ]:
# pipe is called for

city_revenue_tier = (raw_2
                     .pivot_table(index="city", 
                                  columns="revenue_tier",
                                  values="rating",
                                  aggfunc="count",
                                  fill_value=0)
                     .pipe(reverse_columns))

city_revenue_tier

### 9. Sorting

In [ ]:
# Simple sort

raw_2.sort_values("revenue_sek", ascending=False)

In [ ]:
# Subset multiple columns, sort by one of selected columns
 
raw_2.sort_values("revenue_sek", ascending=False)[["restaurant", "city", "revenue_sek", "revenue_tier"]]

In [ ]:
# Subset multiple columns, sort by multiple columns

raw_2.sort_values(["city", "revenue_sek"], ascending=[True, False])[["restaurant", "city", "revenue_sek"]]

### 10. Date & Time

Below are some handy functions for date and time. There are a lot more to learn and practice about date and time. 

In [ ]:
import pandas as pd

df_raw = pd.DataFrame({
    "event": ["A", "B", "C", "D", "E", "F"],
    "date": ["2025-01-15", "2025-06-20", "2025-11-03",
             "2025-03-08", "2025-09-12", "2025-12-24"],
    "date_to_clean": ["15/01/2025", "20-06-2025", "Nov 03 2025",
                      "08.03.2025", "12 Sep 2025", "24/12/2025"],
    "time": ["08:30", "13:45", "17:00",
             "09:15", "14:30", "18:45"]
})

df_raw

In [ ]:
df_raw.dtypes

In [ ]:
df_raw.describe()

#### 1. Parsing dates

In [ ]:
df = df_raw.copy()

# date column is already in standard format — straightforward
df["date"] = pd.to_datetime(df["date"])

df.dtypes

In [ ]:
# date_to_clean has mixed formats — pandas infers automatically

df["date_to_clean"] = pd.to_datetime(df["date_to_clean"], format="mixed", dayfirst=True)

df.dtypes

In [ ]:
df

#### 2. Formatting

Read [Python documentation: `strftime()` and `strptime()` format codes](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes) to try out the format you need. 

In [ ]:
# Convert to a specific string format for display or export

df["date_formatted"] = df["date"].dt.strftime("%d %B %Y")
# 2025-01-15 → "15 January 2025"

df

In [ ]:
df["date_formatted"].dtypes

#### 3. Simple date calculations

In [ ]:
df["eom"] = df["date"] + pd.offsets.MonthEnd(0)
df["eoq"] = df["date"] + pd.offsets.QuarterEnd(0)
df["eoy"] = df["date"] + pd.offsets.YearEnd(0)

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

In [ ]:
df

In [ ]:
df_subset = df[["event", "date", "year","month","day","time","eoy"]]

df_subset

In [ ]:
# How many days until end of year? Here is days remaining after today.

df_subset["days_until_eoy"] = ( df["eoy"] - df["date"]).dt.days

df_subset["days_until_next_eoy"] = (pd.Timestamp("2026-12-31") - df["date"]).dt.days

df_subset

In [ ]:
df_subset = df_subset.drop(columns=["days_until_next_eoy"])

#### 4. Conditional date range — subsetting

In [ ]:
# Filter rows between two dates

df_subset[df_subset["date"].between("2025-01-01", "2025-06-30")]

#### 5. Timestamp

In [ ]:
# Combine date and time columns into one timestamp — needs both columns, so axis=1

df_subset["timestamp"] = df_subset.apply(lambda row: 
                           pd.Timestamp(f"{row['date'].date()} {row['time']}"), axis=1)

df_subset

In [ ]:
# Add a timestamp now

df_subset["processed_at"] = pd.Timestamp.now().strftime("%y%m%d_%H%M%S")

In [ ]:
df_subset

In [ ]:
df_subset.dtypes

### Saving results

Once the data is clean and analysed, it's ready to be saved — to the storage chosen. 

In [ ]:
clean = raw_2.copy()

# Subset to Keep needed columns if desire so
output_cols = ["restaurant", "city", "revenue_sek", "covers", "rev_per_cover", "rating", "revenue_tier"]
result = clean[output_cols].sort_values("revenue_sek", ascending=False)

# Save to CSV
result.to_csv("restaurants_clean.csv", index=False)
print("Saved to restaurants_clean.csv")

# Save to Excel
result.to_excel("restaurants_clean.xlsx", sheet_name="Clean Data", index=False)
print("Saved to restaurants_clean.xlsx")

In [ ]:
# Final result — what we would hand to the report
result

## BREAK TIME 🐍

>Why did the data pipeline go to therapy?<br>
>*Too many bottlenecks.*

## Simulate Data Pipeline

Now we have actually a few new variables. They are tiny and just representations of what we now can use pandas for. 

If we look at the terminal JUPYTER VARIABLES, let's organize them as they are travelling through a medallion architecture:

- source data: raw_data
- bronze layer: raw
- silve/gold layer: raw_2, city_revenue, city_revenue_tier, EXCHANGE_RATE

The silver/gold layer are combined here because different lakehouse design guidlelines might be followed at your work place. Different complexity in projects sometimes also require specific solutions. 

If we put the above into Python scripts to simulate how these steps can work.

In [ ]:
from pathlib import Path

Path.cwd()

In [ ]:
# This can also be run in temrinal.

%run pipeline/run_pipeline.py

In [ ]:
#list(steps)

## Work in a More Pythonic Way

In [ ]:
import sys
import pandas as pd

# To point out where the .py script is stored
# Run ``sys.path`` to see what it contains and what data type the output is.

# below represents scripts/folder
sys.path.append("scripts")
# .append is to add a string to the existing``sys.path``.
# Now it points to scripts folder, where the pipeline_pythonic.py code is stored.

from pipeline_pythonic import load, clean, transform, publish

# Step by step so you can inspect each stage

# In function load(str), the path in string is relative to your notebook's relative path. Therefore, it points to data/ folder
# It has nothing to do with where the above .py script is.
# The custom (user-defined) functions have been imported into in-momery. You can run help(load) to see it.

raw       = load("data/restaurants_raw.csv")
cleaned   = clean(raw)
result    = transform(cleaned)

# output/ here is the 3rd folder path we see in this run .py code block. 
# it points to the output folder.
# all 3 folders we see here are at the same level
publish(result, "output/restaurants_clean.csv")

✓ Loaded 8 rows from data/restaurants_raw.csv
✓ Published 7 rows → output/restaurants_clean.csv


### `assign()` & New Columns

This is one frequently used DataFrame method when chaining up DataFrame in user-defined functions. When used together with `lambda`, it's convenient to create multiple new columns.

The official doc is also very good on this one: https://pandas.pydata.org/pandas-docs/dev/reference/api/pandas.DataFrame.assign.html#pandas.DataFrame.assign


## The Official Pandas Cheat Sheet

There are a lot more useful pandas functions and methods. Above are just a small portion of it. Please do read Wes's book, pandas official doc and of course [Data Wrangling with pandas Cheat Sheet (PDF)](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf).


## Good-to-knows about Scripts

The scripts and workflows we have in our lectures are intentionally created for learning purpose during Nackademin' BI25 - BI-relaterade programspråk. They do not reflect any actual data workflow and they may not uphold to the standard required for Python projects in production.

## END OF LECTURE 🐍

Have a great weekend and see you on Monday!